# Second-model replication

The main Qwen and Llama experiments live in separate artifact roots. This notebook compares methods within each model. Do not treat a raw Qwen–Llama score gap as a method effect; the important question is whether the ordering between methods repeats.

In [ ]:
import os, sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

REPO = Path.cwd().resolve()
while REPO != REPO.parent and not (REPO / 'src' / 'mfr_analysis.py').exists(): REPO = REPO.parent
sys.path.insert(0, str(REPO / 'src'))
import mfr_analysis

PRIMARY = Path(os.environ['MFR_OUTPUT_DIR']).expanduser().resolve()
SECOND = Path(os.environ['MFR_SECOND_OUTPUT_DIR']).expanduser().resolve()
METHODS = ['none', 'random', 'lowest_margin', 'dapr_weak']  # Change only if a different set was preregistered.

In [ ]:
def model_summary(root, label):
    runs = mfr_analysis.load_runs(root, quiet=True, completed_only=True)
    runs = runs[runs.method.astype(str).isin(METHODS)].copy()
    summary = mfr_analysis.summary_table(runs)
    final_rows = []
    for name, part in runs.groupby('run_name', observed=True):
        stage = int(part.stage.max())
        final = part[part.stage.eq(stage)]
        final_rows.append({'run_name': name, 'final_average': final.accuracy.mean()})
    summary = summary.merge(pd.DataFrame(final_rows), on='run_name')
    result = summary.groupby('method', observed=True)[['mean_retention_change', 'final_new_stage_score', 'final_average']].mean().reset_index()
    result.insert(0, 'model', label)
    return result

comparison = pd.concat([model_summary(PRIMARY, 'Qwen2.5-1.5B'), model_summary(SECOND, 'Llama-3.2-3B')], ignore_index=True)
display(comparison.round(2))

In [ ]:
for metric, title in [('mean_retention_change', 'Retention change'), ('final_average', 'Final three-behavior average')]:
    comparison.pivot(index='method', columns='model', values=metric).plot.bar(figsize=(8, 4))
    plt.ylabel(metric.replace('_', ' ')); plt.title(title)
    plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()

## What counts as replication?

The strongest evidence is not identical absolute accuracy. It is the same directional result: the selected method should improve retention and final average over the same baselines on the second model without an unacceptable drop on the current task. Report model-specific results separately, then report whether the method-by-baseline difference has the same sign.